In [6]:
!pip install -q lime reportlab

#!/usr/bin/env python3
"""
CHARUSAT - Responsible and Safe AI (AIML4001) - External Practical, Set B
Explaining Instant EMI-Finance Decisions at an Electronics Showroom

"""
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
CSV_PATH = sys.argv[1] if len(sys.argv) > 1 else "GermanCredit.csv"
OUT_DIR = "."
RANDOM_STATE = 42
PREFERRED_INDEX = 23            # test-set position used in the sample output; falls back to the
                                # first rejected applicant if #23 is not predicted "bad"
SENSITIVE = {"age", "personal_status", "foreign_worker"}
DATASET_NAME = "German Credit (UCI Statlog), 1000 applicants, 20 attributes"

# Column-name aliases so different copies of the dataset all work
ALIASES = {
    "status": "checking_status", "checking_balance": "checking_status", "checking_account": "checking_status",
    "months_loan_duration": "duration", "duration_in_month": "duration",
    "amount": "credit_amount",
    "savings": "savings_status", "savings_balance": "savings_status",
    "employment_duration": "employment", "employment_length": "employment",
    "installment_rate": "installment_commitment",
    "personal_status_sex": "personal_status",
    "other_debtors": "other_parties",
    "present_residence": "residence_since", "residence_history": "residence_since",
    "property": "property_magnitude",
    "other_installment_plans": "other_payment_plans", "installment_plan": "other_payment_plans",
    "number_credits": "existing_credits",
    "people_liable": "num_dependents", "dependents": "num_dependents",
    "telephone": "own_telephone",
}
TARGET_NAMES = ["class", "target", "default", "credit_risk", "risk", "creditability", "label"]

# Readable phrases for the rejection message (feature -> template; {v} = applicant's value)
PHRASES = {
    "checking_status": "the balance in your current/checking account is {v}",
    "duration": "the requested loan duration ({v} months) is long",
    "credit_amount": "the requested credit amount ({v}) is high",
    "savings_status": "your savings level is {v}",
    "purpose": "the purpose of the loan ({v}) carries higher risk",
    "credit_history": "your credit history is '{v}'",
    "employment": "your length of present employment is '{v}'",
    "installment_commitment": "the instalment is a high share of your income (level {v})",
    "property_magnitude": "the property/asset you hold is '{v}'",
    "housing": "your housing situation is '{v}'",
    "other_payment_plans": "you have other payment plans ('{v}')",
    "other_parties": "there is no guarantor/co-applicant ('{v}')",
    "existing_credits": "you already have {v} existing credit(s)",
    "job": "your job category is '{v}'",
    "residence_since": "your time at the present residence is {v}",
    "own_telephone": "no registered telephone ('{v}')",
    "num_dependents": "number of dependents is {v}",
}


def out(name):
    return os.path.join(OUT_DIR, name)


LOG = []


def say(text=""):
    print(text)
    LOG.append(text)


# ----------------------------------------------------------------------------
# TASK 1 : load, encode, split
# ----------------------------------------------------------------------------
def load_data(path):
    if not os.path.exists(path):
        sys.exit(f"File '{path}' not found. Put GermanCredit.csv in your folder.")
    df = pd.read_csv(path)
    if df.shape[1] == 1 or df.columns[0] == "A1":      # unlikely, but cover odd separators
        df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [str(c).strip().strip('"') for c in df.columns]
    low = {c: c.lower().strip() for c in df.columns}
    df = df.rename(columns=low)
    df = df.rename(columns={c: ALIASES.get(c, c) for c in df.columns})

    tcol = next((c for c in TARGET_NAMES if c in df.columns), None)
    if tcol is None:
        tcol = df.columns[-1]
    t = df[tcol]
    if t.dtype == object or str(t.dtype).startswith("str"):
        y = (t.astype(str).str.strip().str.lower().isin(["bad", "2", "yes", "default"])).astype(int)
    else:
        vals = set(pd.unique(t))
        if vals <= {1, 2}:                 # UCI coding: 1 = good, 2 = bad
            y = (t == 2).astype(int)
        elif vals <= {0, 1}:
            # 'default' style columns: 1 = defaulted (bad); 'creditability/credit_risk': 1 = good
            y = (t == 1).astype(int) if tcol in ("default", "target", "label") else (t == 0).astype(int)
        else:
            sys.exit(f"Cannot interpret target column '{tcol}' with values {vals}")
    X = df.drop(columns=[tcol]).copy()
    return X, y.rename("bad"), tcol


# Overwrite CSV_PATH in case it was set to internal notebook args like '-f'
if not os.path.exists(CSV_PATH) and os.path.exists("GermanCredit.csv"):
    CSV_PATH = "GermanCredit.csv"

X_raw, y, tcol = load_data(CSV_PATH)
cat_cols = [c for c in X_raw.columns if not pd.api.types.is_numeric_dtype(X_raw[c])]
num_cols = [c for c in X_raw.columns if c not in cat_cols]

enc = OrdinalEncoder()                      # one feature per original attribute
X = X_raw.copy()
X[cat_cols] = enc.fit_transform(X_raw[cat_cols].astype(str))
X = X.astype(float)
feature_names = list(X.columns)
cat_idx = [feature_names.index(c) for c in cat_cols]
cat_names = {feature_names.index(c): list(cats) for c, cats in zip(cat_cols, enc.categories_)}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_train_raw, X_test_raw = X_raw.loc[X_train.index], X_raw.loc[X_test.index]

say("=" * 70)
say("TASK 1 : DATA")
say(f"Dataset      : {DATASET_NAME}  (file: {os.path.basename(CSV_PATH)})")
say(f"Shape        : {X_raw.shape[0]} rows x {X_raw.shape[1]} features, target column = '{tcol}'")
say(f"Categorical  : {len(cat_cols)} (OrdinalEncoder)   Numeric: {len(num_cols)}")
say(f"Class balance: bad(1) = {int(y.sum())}, good(0) = {int((y == 0).sum())}")
say(f"Split (80:20, stratified, random_state=42): train = {len(X_train)}, test = {len(X_test)}")

# ----------------------------------------------------------------------------
# TASK 2 : Random Forest
# ----------------------------------------------------------------------------
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
proba_test = rf.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)
acc = accuracy_score(y_test, pred_test)
auc = roc_auc_score(y_test, proba_test)

say("\n" + "=" * 70)
say("TASK 2 : MODEL AND GLOBAL EXPLANATION")
say("Model : RandomForest (200 trees, depth 8)")
say(f"Test accuracy : {acc:.2f}      ROC-AUC : {auc:.2f}")

# ----------------------------------------------------------------------------
# TASK 3 : SHAP global importance
# ----------------------------------------------------------------------------
explainer = shap.TreeExplainer(rf)
sv_raw = explainer.shap_values(X_test)


def bad_class_values(sv):
    """Return (n_samples, n_features) SHAP values for class 1 (= bad) across shap versions."""
    if isinstance(sv, list):
        return np.asarray(sv[1])
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv


sv = bad_class_values(sv_raw)
ev = explainer.expected_value
base_value = float(np.ravel(ev)[1] if np.size(ev) > 1 else np.ravel(ev)[0])

mean_abs = np.abs(sv).mean(axis=0)
order = np.argsort(-mean_abs)
top5 = [(feature_names[i], mean_abs[i]) for i in order[:5]]

say("\nGLOBAL FEATURE IMPORTANCE (mean |SHAP|, class = bad)")
say("RANK  FEATURE                 MEAN|SHAP|")
say("-" * 42)
for r, (f, v) in enumerate(top5, 1):
    say(f" {r:<4} {f:<24} {v:.3f}")
say("-" * 42)

plt.figure()
shap.summary_plot(sv, X_test, feature_names=feature_names, show=False)
plt.title("SHAP summary (beeswarm) - class = bad", fontsize=10)
plt.tight_layout()
plt.savefig(out("shap_summary.png"), dpi=150, bbox_inches="tight")
plt.close()

plt.figure()
shap.summary_plot(sv, X_test, feature_names=feature_names, plot_type="bar", show=False)
plt.title("Global importance: mean |SHAP| - class = bad", fontsize=10)
plt.tight_layout()
plt.savefig(out("shap_bar.png"), dpi=150, bbox_inches="tight")
plt.close()
say("Plot saved : shap_summary.png (and shap_bar.png)")

# ----------------------------------------------------------------------------
# TASK 4 : SHAP waterfall for one rejected applicant
# ----------------------------------------------------------------------------
rejected_pos = [i for i in range(len(X_test)) if pred_test[i] == 1]
if not rejected_pos:
    sys.exit("No applicant in the test set is predicted 'bad'.")
pos = PREFERRED_INDEX if (PREFERRED_INDEX < len(X_test) and pred_test[PREFERRED_INDEX] == 1) else rejected_pos[0]
row_raw = X_test_raw.iloc[pos]
row_enc = X_test.iloc[pos].values
p_bad = proba_test[pos]


def readable(fname, enc_val):
    """Original (human-readable) value of a feature for this applicant."""
    if fname in cat_cols:
        return row_raw[fname]
    v = row_raw[fname]
    return int(v) if float(v).is_integer() else round(float(v), 2)


def shap_reasons(k=None):
    idx = np.argsort(-sv[pos])             # most positive first = pushes towards 'bad'
    idx = [i for i in idx if sv[pos][i] > 0]
    return idx if k is None else idx[:k]


top3_idx = shap_reasons(3)
say("\n" + "=" * 70)
say(f"TASK 4 : SHAP LOCAL EXPLANATION")
say(f"Applicant (test index {pos}) : P(bad) = {p_bad:.2f}  ->  "
    f"{'REJECTED' if p_bad >= 0.5 else 'APPROVED'}   (actual label: {'bad' if y_test.iloc[pos] else 'good'})")
say(f"SHAP base value (expected P(bad)) = {base_value:.3f}")
say("Top 3 SHAP reasons (positive = pushes towards REJECTION):")
for i in top3_idx:
    say(f"   {feature_names[i]}={readable(feature_names[i], row_enc[i])}   {sv[pos][i]:+.3f}")

expl = shap.Explanation(values=sv[pos], base_values=base_value,
                        data=np.array([str(readable(f, v)) for f, v in zip(feature_names, row_enc)], dtype=object),
                        feature_names=feature_names)
plt.figure()
shap.plots.waterfall(expl, max_display=10, show=False)
plt.title(f"SHAP waterfall - test applicant {pos} (P(bad) = {p_bad:.2f})", fontsize=10)
plt.savefig(out("shap_waterfall.png"), dpi=150, bbox_inches="tight")
plt.close()
say("Plot saved : shap_waterfall.png")

# ----------------------------------------------------------------------------
# TASK 5 : LIME + comparison
# ----------------------------------------------------------------------------
lime_exp = LimeTabularExplainer(
    X_train.values, feature_names=feature_names, class_names=["good", "bad"],
    categorical_features=cat_idx, categorical_names=cat_names,
    discretize_continuous=True, mode="classification", random_state=RANDOM_STATE)
exp = lime_exp.explain_instance(row_enc, rf.predict_proba, num_features=5, labels=(1,),
                                num_samples=5000)
lime_pairs = exp.as_list(label=1)           # (text, weight)
lime_map = exp.as_map()[1]                  # (feature_index, weight)
lime_feats = [feature_names[i] for i, _ in lime_map]
lime_feats_sorted = [feature_names[i] for i, w in sorted(lime_map, key=lambda t: -abs(t[1]))]

fig = exp.as_pyplot_figure(label=1)
plt.tight_layout()
fig.savefig(out("lime_explanation.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

shap_top3_names = [feature_names[i] for i in top3_idx]
agree_in_lime5 = [f for f in shap_top3_names if f in lime_feats]
agree_top3 = [f for f in shap_top3_names if f in lime_feats_sorted[:3]]

say("\n" + "=" * 70)
say("TASK 5 : LIME (num_features = 5) vs SHAP")
say(f"{'SHAP (top 3)':<38}{'LIME (top 5)'}")
say("-" * 78)
for k in range(5):
    left = ""
    if k < len(top3_idx):
        i = top3_idx[k]
        left = f" {feature_names[i]}={readable(feature_names[i], row_enc[i])}  {sv[pos][i]:+.3f}"
    right = f"{lime_pairs[k][0]}  {lime_pairs[k][1]:+.2f}" if k < len(lime_pairs) else ""
    say(f"{left:<38}{right}")
say("-" * 78)
say(f"Top-3 agreement (SHAP top-3 found in LIME top-5)  : {len(agree_in_lime5)} / 3  {agree_in_lime5}")
say(f"Strict agreement (SHAP top-3 vs LIME top-3 by |w|): {len(agree_top3)} / 3  {agree_top3}")
say("Positive LIME weight = pushes towards 'bad'; negative = pushes towards 'good'.")
say("Plot saved : lime_explanation.png")

# ----------------------------------------------------------------------------
# TASK 6 : rejection message + sensitive-attribute check
# ----------------------------------------------------------------------------
all_reason_feats = set(shap_top3_names) | {f for f in lime_feats}
sensitive_found = sorted(f for f in all_reason_feats if f in SENSITIVE)

legit = [i for i in shap_reasons() if feature_names[i] not in SENSITIVE][:2]
parts = []
for i in legit:
    f = feature_names[i]
    v = readable(f, row_enc[i])
    tmpl = PHRASES.get(f, f.replace("_", " ") + " ({v})")
    parts.append(tmpl.format(v=v))
reasons_text = " and ".join(parts) if parts else "the overall risk assessment of your application"
message = (f"Your EMI request was not approved mainly because {reasons_text}. "
           f"You are welcome to re-apply after improving these factors or to choose a smaller amount/shorter plan.")

say("\n" + "=" * 70)
say("TASK 6 : REJECTION MESSAGE AND SENSITIVE-ATTRIBUTE CHECK")
if sensitive_found:
    say(f"Sensitive attributes in reasons : {', '.join(sensitive_found)} -> REVIEW")
else:
    say("Sensitive attributes in reasons : none -> OK")
say(f'Message (legitimate reasons only): "{message}"')

# ----------------------------------------------------------------------------
# Save results.txt
# ----------------------------------------------------------------------------
with open(out("results.txt"), "w", encoding="utf-8") as fh:
    fh.write("\n".join(LOG))

# ----------------------------------------------------------------------------
# Optional: auto-generate the PDF report (aim, algorithm, program, output, conclusion, Q1-Q4)
# ----------------------------------------------------------------------------
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Image, Preformatted,
                                    PageBreak)
    from xml.sax.saxutils import escape

    ss = getSampleStyleSheet()
    body = ParagraphStyle("b", parent=ss["BodyText"], fontSize=10, leading=14)
    code = ParagraphStyle("c", parent=ss["Code"], fontSize=7.5, leading=9)
    H = ss["Heading2"]
    S = []
    P = lambda t, st=body: S.append(Paragraph(t, st))

    S.append(Paragraph("Responsible and Safe AI (AIML4001) - External Practical, Set B", ss["Title"]))
    P("<b>Explaining Instant EMI-Finance Decisions at an Electronics Showroom</b>")
    P(f"Dataset used: {escape(DATASET_NAME)}")

    S.append(Paragraph("1. Aim", H))
    P("To build an explainability pipeline for an instant-EMI credit model: train a Random Forest, "
      "explain it globally with SHAP, explain one rejected applicant locally with SHAP and LIME, "
      "and generate a plain-language rejection reason while checking for sensitive attributes.")

    S.append(Paragraph("2. Algorithm", H))
    for t in ["Load the credit data; treat 'bad' as class 1; encode categorical columns with OrdinalEncoder "
              "(one feature per attribute); stratified 80:20 split with random_state=42.",
              "Train RandomForestClassifier(n_estimators=200, max_depth=8); report test accuracy and ROC-AUC.",
              "Compute SHAP values with TreeExplainer (class = bad); mean |SHAP| gives global ranking; "
              "draw beeswarm and bar plots.",
              "Pick a rejected test applicant (P(bad) >= 0.5); draw SHAP waterfall; top 3 positive SHAP = reasons.",
              "Explain the same applicant with LIME (num_features=5); compare with SHAP top 3.",
              "Create a rejection message from non-sensitive top reasons; flag age, personal_status, foreign_worker."]:
        P("&bull; " + escape(t))

    S.append(Paragraph("3. Program", H))
    P("Full source: german_credit_explainability.py (included below).")
    try:
        # Fix NameError when __file__ is missing in notebook context
        script_file = "german_credit_explainability.py"
        if os.path.exists(script_file):
            src = open(script_file, encoding="utf-8").read()
        else:
            src = "# Running interactively inside a Jupyter / Colab notebook cell."
        S.append(Preformatted(src, code))
    except Exception as e:
        pass

    S.append(PageBreak())
    S.append(Paragraph("4. Output", H))
    S.append(Preformatted("\n".join(LOG), ParagraphStyle("o", parent=code, fontSize=7.5)))
    for fn, cap in [("shap_summary.png", "SHAP summary (beeswarm)"), ("shap_bar.png", "Global mean |SHAP|"),
                    ("shap_waterfall.png", "SHAP waterfall - rejected applicant"),
                    ("lime_explanation.png", "LIME explanation - same applicant")]:
        if os.path.exists(out(fn)):
            P(f"<b>{cap}</b>")
            im = Image(out(fn))
            r = min(15 * cm / im.imageWidth, 10 * cm / im.imageHeight)
            im.drawWidth, im.drawHeight = im.imageWidth * r, im.imageHeight * r
            S.append(im)
            S.append(Spacer(1, 6))

    S.append(Paragraph("5. Conclusion", H))
    P(f"The Random Forest reached accuracy {acc:.2f} and ROC-AUC {auc:.2f}. SHAP shows the main global drivers of "
      f"rejection are {', '.join(f for f, _ in top5)}. Local SHAP and LIME explanations give applicant-specific "
      f"reasons that can be converted into a clear, fair-lending-compliant message, provided sensitive attributes "
      f"are reviewed and excluded from the stated reasons.")

    S.append(Paragraph("6. Technical Questions", H))
    top5_txt = ", ".join(f"{f} ({v:.3f})" for f, v in top5)
    shap_txt = "; ".join(f"{feature_names[i]}={readable(feature_names[i], row_enc[i])} ({sv[pos][i]:+.3f})"
                         for i in top3_idx)
    lime_txt = "; ".join(f"{t} ({w:+.2f})" for t, w in lime_pairs)
    sens_txt = (f"Yes - {', '.join(sensitive_found)} appeared among the SHAP/LIME reasons."
                if sensitive_found else "No sensitive attribute appeared among the reasons.")
    QA = [
        ("Q1. Accuracy, ROC-AUC, top-5 features; what does the global explanation tell the NBFC?",
         f"Test accuracy = {acc:.2f}, ROC-AUC = {auc:.2f}. Top 5 features by mean |SHAP|: {top5_txt}. "
         f"The global explanation shows which factors the model relies on most across all applicants "
         f"(chiefly {top5[0][0]} and {top5[1][0]}). The NBFC can check that these drivers are financially "
         f"sensible, spot reliance on proxies for sensitive traits, document the model for compliance, "
         f"and design clearer customer guidance."),
        ("Q2. Top 3 SHAP reasons for the rejected applicant; meaning of a positive SHAP value?",
         f"Applicant (test index {pos}), P(bad) = {p_bad:.2f}. Top 3 SHAP reasons: {shap_txt}. "
         f"In the waterfall plot a positive SHAP value means that the feature's value pushes the prediction "
         f"higher than the base value, i.e. towards class 'bad' (more likely to be rejected); negative values "
         f"push towards 'good'. The bar length shows how strongly."),
        ("Q3. Compare LIME and SHAP; how many of the top 3 agree and why can they differ?",
         f"LIME (5 features): {lime_txt}. {len(agree_in_lime5)} of the 3 SHAP top reasons also appear in LIME's "
         f"top 5 ({', '.join(agree_in_lime5) or 'none'}). They can differ because SHAP uses exact Shapley "
         f"attributions from the tree structure (consistent, additive), whereas LIME fits a local linear "
         f"surrogate on random perturbations around the applicant, with discretised bins and random sampling, "
         f"so results depend on the neighbourhood, binning and sampling noise."),
        ("Q4. Rejection message; sensitive attributes; what should the NBFC do?",
         f'Message: "{message}" {sens_txt} The NBFC should not cite sensitive attributes (age, marital/personal '
         f'status, foreign-worker status) as reasons; it should give only legitimate financial reasons, review '
         f'or remove such features and their proxies, run fairness audits (e.g. approval-rate and error-rate '
         f'gaps across groups), and provide human review/appeal and documentation for fair-lending compliance.'),
    ]
    for q, a in QA:
        P(f"<b>{escape(q)}</b>")
        P(escape(a))
        S.append(Spacer(1, 6))

    SimpleDocTemplate(out("report.pdf"), pagesize=A4, leftMargin=1.8 * cm, rightMargin=1.8 * cm,
                      topMargin=1.8 * cm, bottomMargin=1.8 * cm).build(S)
    print("PDF report saved : report.pdf  (add your name/enrolment no. before submitting)")
except ImportError:
    print("(reportlab not installed -> skipped auto PDF; use results.txt + PNGs for your report)")

TASK 1 : DATA
Dataset      : German Credit (UCI Statlog), 1000 applicants, 20 attributes  (file: GermanCredit.csv)
Shape        : 1000 rows x 20 features, target column = 'credit_risk'
Categorical  : 13 (OrdinalEncoder)   Numeric: 7
Class balance: bad(1) = 300, good(0) = 700
Split (80:20, stratified, random_state=42): train = 800, test = 200

TASK 2 : MODEL AND GLOBAL EXPLANATION
Model : RandomForest (200 trees, depth 8)
Test accuracy : 0.75      ROC-AUC : 0.79

GLOBAL FEATURE IMPORTANCE (mean |SHAP|, class = bad)
RANK  FEATURE                 MEAN|SHAP|
------------------------------------------
 1    checking_status          0.089
 2    duration                 0.043
 3    savings_status           0.032
 4    credit_amount            0.027
 5    other_payment_plans      0.020
------------------------------------------
Plot saved : shap_summary.png (and shap_bar.png)

TASK 4 : SHAP LOCAL EXPLANATION
Applicant (test index 23) : P(bad) = 0.60  ->  REJECTED   (actual label: bad)
SHAP bas